# 🤖 AI Resume Screening System
### Built with LangChain + LangSmith | GenAI Internship Assignment

**Pipeline:** Resume → Skill Extraction → Matching → Scoring → Explanation → LangSmith Traces

---

## 1. Environment Setup & LangSmith Configuration

In [ ]:
# Install dependencies (run once)
# !pip install langchain langchain-openai langsmith python-dotenv openai

import os
from dotenv import load_dotenv

# Load from .env file
load_dotenv()

# ── OR set directly here for Jupyter ──────────────────────────────────────
# os.environ['OPENAI_API_KEY']        = 'sk-your-openai-key'
# os.environ['LANGCHAIN_API_KEY']     = 'ls__your-langsmith-key'
# os.environ['LANGCHAIN_TRACING_V2']  = 'true'
# os.environ['LANGCHAIN_PROJECT']     = 'ai-resume-screening'

# Verify setup
tracing = os.getenv('LANGCHAIN_TRACING_V2', 'false')
project = os.getenv('LANGCHAIN_PROJECT', 'default')
print(f"LangSmith Tracing: {'✅ ENABLED' if tracing == 'true' else '❌ DISABLED'}")
print(f"Project: {project}")
print(f"OpenAI Key set: {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")

## 2. Import Pipeline Modules

In [ ]:
from langchain_openai import ChatOpenAI
from langsmith import traceable

# Import our modular components
from prompts.resume_prompts import (
    skill_extraction_prompt,
    matching_prompt,
    scoring_prompt,
    explanation_prompt,
)
from chains.screening_chains import ResumeScreeningPipeline
from data.sample_data import JOB_DESCRIPTION, CANDIDATES, RESUME_STRONG, RESUME_AVERAGE, RESUME_WEAK

print('✅ All modules imported successfully')

## 3. Initialize LLM + Pipeline

In [ ]:
# Initialize LLM
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,       # Deterministic for consistent scoring
    max_tokens=2000,
)

# Initialize full pipeline
pipeline = ResumeScreeningPipeline(llm=llm)

print('✅ LLM and Pipeline initialized')
print(f'   Model: {llm.model_name}')

## 4. View Job Description

In [ ]:
print('JOB DESCRIPTION')
print('=' * 60)
print(JOB_DESCRIPTION)

## 5. Run 1: Strong Candidate (Aisha Patel)

In [ ]:
@traceable(name='screen_strong_candidate', tags=['strong', 'run-1'])
def run_strong():
    return pipeline.run(
        resume=RESUME_STRONG,
        job_description=JOB_DESCRIPTION,
        candidate_name='Aisha Patel'
    )

result_strong = run_strong()
print(f"\n📊 Score: {result_strong['overall_score']}/100")
print(f"   Category: {result_strong['score_category']}")
print(f"   Decision: {result_strong['recommendation']}")

In [ ]:
# View extracted profile
import json
print('EXTRACTED PROFILE:')
print(json.dumps(result_strong['extracted_profile'], indent=2))

In [ ]:
# View explanation report
print('EVALUATION REPORT:')
print('=' * 60)
print(result_strong['explanation'])

## 6. Run 2: Average Candidate (Rahul Sharma)

In [ ]:
@traceable(name='screen_average_candidate', tags=['average', 'run-2'])
def run_average():
    return pipeline.run(
        resume=RESUME_AVERAGE,
        job_description=JOB_DESCRIPTION,
        candidate_name='Rahul Sharma'
    )

result_average = run_average()
print(f"\n📊 Score: {result_average['overall_score']}/100")
print(f"   Category: {result_average['score_category']}")
print(f"   Decision: {result_average['recommendation']}")

In [ ]:
print('EVALUATION REPORT:')
print('=' * 60)
print(result_average['explanation'])

## 7. Run 3: Weak Candidate (Vikram Singh)

In [ ]:
@traceable(name='screen_weak_candidate', tags=['weak', 'run-3'])
def run_weak():
    return pipeline.run(
        resume=RESUME_WEAK,
        job_description=JOB_DESCRIPTION,
        candidate_name='Vikram Singh'
    )

result_weak = run_weak()
print(f"\n📊 Score: {result_weak['overall_score']}/100")
print(f"   Category: {result_weak['score_category']}")
print(f"   Decision: {result_weak['recommendation']}")

In [ ]:
print('EVALUATION REPORT:')
print('=' * 60)
print(result_weak['explanation'])

## 8. LangSmith Debug Run — Hallucination Guard Test

In [ ]:
# Test pipeline with a tricky resume (buzzwords but no real skills)
# This verifies the pipeline does NOT hallucinate skills

tricky_resume = """
Name: Buzz Wordsmith
Objective: Passionate about AI, Machine Learning, Big Data, and Cloud Computing.
Interested in becoming a Data Scientist. Watched many online courses.
Skills (self-assessed): AI, ML, Deep Learning, Python, TensorFlow (beginner)
Education: B.A. English Literature, 2023
Experience: None
"""

@traceable(name='debug_hallucination_guard', tags=['debug', 'edge-case'])
def run_debug():
    return pipeline.run(
        resume=tricky_resume,
        job_description=JOB_DESCRIPTION,
        candidate_name='Buzz Wordsmith (Debug)'
    )

result_debug = run_debug()
score = result_debug.get('overall_score', 'N/A')
print(f'Debug Score: {score}/100')

# Score should be very low (<25) — if not, check LangSmith trace for prompt issues
if isinstance(score, int) and score > 25:
    print('⚠️  Score seems inflated — check LangSmith trace for hallucination!')
else:
    print('✅ Hallucination guard working — score is correctly low')

## 9. Final Comparison Summary

In [ ]:
results = [result_strong, result_average, result_weak]
sorted_results = sorted(results, key=lambda r: r.get('overall_score', 0) or 0, reverse=True)

print('FINAL CANDIDATE RANKING')
print('=' * 60)
print(f"{'Rank':<5} {'Candidate':<20} {'Score':>6} {'Decision'}")
print('-' * 60)

for rank, r in enumerate(sorted_results, 1):
    print(f"{rank:<5} {r['candidate_name']:<20} {str(r['overall_score']):>6}   {r['recommendation']}")

print('\n📊 View full traces at: https://smith.langchain.com')

## 10. Score Visualization
> Simple bar chart of candidate scores

In [ ]:
import matplotlib.pyplot as plt

names  = [r['candidate_name'] for r in results]
scores = [r.get('overall_score', 0) or 0 for r in results]
colors = ['#2ecc71', '#f39c12', '#e74c3c']  # green, orange, red

plt.figure(figsize=(10, 5))
bars = plt.bar(names, scores, color=colors, edgecolor='black', linewidth=0.8)

# Add score labels on bars
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{score}/100', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 110)
plt.ylabel('Fit Score (0-100)', fontsize=12)
plt.title('AI Resume Screening — Candidate Scores', fontsize=14, fontweight='bold')
plt.axhline(y=60, color='gray', linestyle='--', alpha=0.5, label='Minimum Threshold (60)')
plt.legend()
plt.tight_layout()
plt.savefig('screenshots/score_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to screenshots/score_chart.png')